# Ticket Data Analysis for PoC Scenarios
## De lokale troubleshooter - Ultimum MSP

**Doel:** Analyseer de historische ticketdata om de meest geschikte repetitieve scenario's te identificeren voor de Proof of Concept van de lokale AI troubleshooting agent.

**Focus:** 
- Hoogfrequente, laag-complexe issues
- Geschikt voor diagnose op basis van titel + logs (secure by design)
- Geschikt voor Human-in-the-Loop validatie
- Level 1 Servicedesk tickets (meeste ruis)

In [9]:
# Imports
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re

# Settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries loaded successfully")

Libraries loaded successfully


In [10]:
# Load the anonymized ticket data
df = pd.read_csv('anonsecondfinal_real.csv', low_memory=False)

print(f"Total tickets: {len(df)}")
print(f"\nColumns: {df.columns.tolist()}")

Total tickets: 3331

Columns: ['Ticket Number', 'Title', 'Queue', 'Issue Type', 'Sub-Issue Type', 'Priority', 'Created', 'Due', 'Total Hours Worked', 'Contract Type', 'Resolved Time', 'Source', 'Ticket Category', 'Ticket Type', 'Work Type']


## 1. Algemene Overzicht van de Data

We kijken naar de verdeling van Issue Types en Sub-Issue Types om te zien waar de meeste tickets zitten.

In [11]:
print("=== Issue Type Distribution (Top 10) ===")
print(df['Issue Type'].value_counts().head(10))

print("\n=== Sub-Issue Type Distribution (Top 15) ===")
print(df['Sub-Issue Type'].value_counts().head(15))

print("\n=== Ticket Category ===")
print(df['Ticket Category'].value_counts())

print("\n=== Queue Distribution ===")
print(df['Queue'].value_counts())

=== Issue Type Distribution (Top 10) ===
Issue Type
SR - User management            705
SR - Software & Applications    557
IM - Maintenance                390
SR - Microsoft 365              245
SR - Workstation                186
IM - Network                    143
IM - Workstation                130
IM - Security                   115
SR - Maintenance                112
SR - Network                     99
Name: count, dtype: int64

=== Sub-Issue Type Distribution (Top 15) ===
Sub-Issue Type
Onboarding user                   205
Other                             202
Exchange online                   193
Permissions                       175
Offboarding user                  173
Regular maintenance               108
Password/MFA reset                106
Internet                          102
Install/reinstall                  81
Configuration                      69
Security vulnerability/malware     63
Performance                        62
User details & settings            46
VPN    

## 2. Keyword Analyse uit Ticket Titels

Dit geeft inzicht in de meest voorkomende thema's in de ticketbeschrijvingen.

In [12]:
# Extract relevant keywords from titles
titles = df['Title'].dropna().astype(str).str.lower()

target_keywords = ['mfa', 'reset', 'onboarding', 'offboarding', 'shared', 'mailbox', 
                   'disk', 'space', 'vpn', 'login', 'password', 'outlook', 'avd', 
                   'performance', 'slow', 'install', 'permission', 'exchange', 'traagheid']

keyword_counts = Counter()
for title in titles:
    for kw in target_keywords:
        if kw in title:
            keyword_counts[kw] += 1

print("=== Top Keywords in Ticket Titles ===")
for kw, count in keyword_counts.most_common(15):
    print(f"{kw:15} : {count:4d} tickets")

=== Top Keywords in Ticket Titles ===
onboarding      :  199 tickets
offboarding     :  141 tickets
install         :  135 tickets
reset           :  116 tickets
mailbox         :  107 tickets
shared          :   71 tickets
permission      :   69 tickets
login           :   59 tickets
mfa             :   57 tickets
vpn             :   57 tickets
password        :   55 tickets
outlook         :   42 tickets
space           :   12 tickets
avd             :    8 tickets
disk            :    7 tickets


## 3. Diepgaande Analyse van Potentiële Scenario's

We analyseren de top kandidaten voor de PoC op basis van frequentie en geschiktheid voor automatisering.

In [13]:
# Analyze top scenarios

scenarios = {
    'MFA / Password Reset': df['Title'].str.contains('MFA|Password reset', case=False, na=False),
    #'Onboarding User': df['Title'].str.contains('Onboarding', case=False, na=False),
    #'Offboarding User': df['Title'].str.contains('Offboarding', case=False, na=False),
    #'Shared Mailbox': df['Title'].str.contains('shared mailbox', case=False, na=False),
    'VPN Issues': df['Title'].str.contains('VPN', case=False, na=False),
    'Login Issues': df['Title'].str.contains('login|inlog', case=False, na=False),
    'Disk Space': df['Title'].str.contains('disk|schijf|space', case=False, na=False),
    'Performance / Traag': df['Title'].str.contains('traag|slow|performance', case=False, na=False),
}

print("=== Potentiële PoC Scenario's - Frequentie ===\n")
for name, mask in scenarios.items():
    count = mask.sum()
    pct = (count / len(df)) * 100
    print(f"{name:25} : {count:4d} tickets ({pct:5.1f}%)")

=== Potentiële PoC Scenario's - Frequentie ===

MFA / Password Reset      :  107 tickets (  3.2%)
VPN Issues                :   57 tickets (  1.7%)
Login Issues              :   96 tickets (  2.9%)
Disk Space                :   51 tickets (  1.5%)
Performance / Traag       :   17 tickets (  0.5%)


## 4. Aanbeveling voor Beste Scenario's (PoC)

Op basis van de data en de eisen uit het PvA (repetitief, laag-complex, diagnoseerbaar, Human-in-the-Loop geschikt):

### Top 3 aanbevolen scenario's voor de Proof of Concept:

1. **MFA / Password Reset** ( ~110+ tickets )
   - Zeer frequent
   - Eenvoudige diagnose op basis van titel + bron (Telephone/Email)
   - Standaard procedure → perfect voor agent die stappen voorstelt
   - Laag risico, Human-in-the-Loop makkelijk

2. **User Onboarding** ( ~200 tickets )
   - Hoogste volume
   - Gestructureerd proces (accounts aanmaken, groepen, MFA, etc.)
   - Agent kan checklist genereren en validatie doen
   - Uitstekend voor demonstratie van end-to-end automatisering

3. **Shared Mailbox / Permissions** of **Offboarding**
   - Offboarding ook ~140 tickets
   - Shared mailbox issues komen vaak voor
   - Goed combineerbaar met user management

### Alternatief / 4e scenario:
- **Disk Space / Maintenance alerts** (via monitoring)
- **VPN / Login issues**

Deze scenario's passen perfect bij:
- Secure-by-design (geen PII naar LLM)
- Human-in-the-Loop
- Lokale LLM kan context uit titel + eventuele logs gebruiken
- Vermindert significante ruis op Level 1 Servicedesk

In [14]:
# Focus on Level 1 Servicedesk (most repetitive noise)
level1 = df[df['Queue'].str.contains('Level 1', case=False, na=False)]
print(f"Level 1 tickets: {len(level1)} ({len(level1)/len(df)*100:.1f}% of total)")

print("\nTop Issue Types in Level 1:")
print(level1['Issue Type'].value_counts().head(8))

Level 1 tickets: 2850 (85.6% of total)

Top Issue Types in Level 1:
Issue Type
SR - User management            643
SR - Software & Applications    364
IM - Maintenance                355
SR - Microsoft 365              216
SR - Workstation                169
IM - Workstation                128
IM - Network                    126
IM - Security                   109
Name: count, dtype: int64


## 5. Tijdsinvestering per Scenario

Kijk welke issues relatief veel tijd kosten ondanks dat ze repetitief zijn.

In [15]:
# Average hours worked per scenario
print("=== Gemiddelde uren gewerkt per scenario ===\n")

for name, mask in scenarios.items():
    subset = df[mask]
    if len(subset) > 0:
        avg_hours = subset['Total Hours Worked'].mean()
        total_hours = subset['Total Hours Worked'].sum()
        print(f"{name:25} : {avg_hours:.2f} uur gemiddeld | Totaal: {total_hours:.1f} uur")

=== Gemiddelde uren gewerkt per scenario ===

MFA / Password Reset      : 0.36 uur gemiddeld | Totaal: 38.5 uur
VPN Issues                : 0.88 uur gemiddeld | Totaal: 50.0 uur
Login Issues              : 0.48 uur gemiddeld | Totaal: 46.0 uur
Disk Space                : 0.54 uur gemiddeld | Totaal: 27.8 uur
Performance / Traag       : 0.93 uur gemiddeld | Totaal: 15.8 uur
